In [1]:
!pip install scikit-learn pandas joblib flask

In [12]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import joblib

# ============================================
# GENERATE 100 POSITIVE + 100 NEGATIVE USING LOOPS
# ============================================

positive_words = ['amazing', 'love', 'best', 'great', 'good', 'excellent', 'fantastic',
                  'wonderful', 'perfect', 'brilliant', 'awesome', 'happy', 'satisfied',
                  'impressed', 'outstanding', 'exceptional', 'smooth', 'fast', 'reliable',
                  'durable', 'easy', 'beautiful', 'sleek', 'powerful', 'premium', 'clear',
                  'sharp', 'stunning', 'rich', 'solid', 'comfortable', 'stylish', 'responsive',
                  'helpful', 'quick', 'simple', 'true', 'better', 'glad', 'enjoy',
                  'recommend', 'worth', 'deal', 'value', 'quality', 'performance', 'design',
                  'features', 'packaging', 'shipping', 'delivery', 'support', 'service',
                  'instructions', 'setup', 'protected', 'arrived', 'flawless', 'stunning',
                  'rich', 'perfect', 'lightweight', 'solid', 'professional', 'boosted',
                  'saved', 'enjoy', 'love', 'like', 'happy', 'pleased', 'delighted',
                  'thrilled', 'excited', 'grateful', 'appreciate', 'admire', 'respect',
                  'praise', 'compliment', 'approve', 'endorse', 'favor', 'prefer', 'choose',
                  'select', 'pick', 'adopt', 'embrace', 'welcome', 'celebrate', 'cheer',
                  'applaud', 'commend', 'recommend', 'suggest', 'propose', 'advocate', 'support']

negative_words = ['terrible', 'waste', 'bad', 'poor', 'disappointed', 'hate', 'worst',
                  'broken', 'horrible', 'awful', 'defective', 'cheap', 'stopped', 'regret',
                  'slow', 'ugly', 'uncomfortable', 'difficult', 'confusing', 'weak', 'noisy',
                  'overheating', 'flickering', 'crashes', 'laggy', 'misleading', 'false',
                  'rude', 'late', 'damaged', 'missing', 'wrong', 'small', 'expensive',
                  'outdated', 'limited', 'unreliable', 'disaster', 'junk', 'garbage', 'trash',
                  'pathetic', 'nightmare', 'angry', 'furious', 'scam', 'fraud', 'fake',
                  'failed', 'dead', 'counterfeit', 'rusty', 'loose', 'leaking', 'burning',
                  'blurred', 'distorted', 'charging', 'bluetooth', 'wifi', 'app', 'software',
                  'bugs', 'abandoned', 'ignoring', 'unprofessional', 'response', 'hidden',
                  'overpriced', 'cheaper', 'loot', 'worthless', 'useless', 'disgusting',
                  'annoying', 'irritating', 'frustrating', 'boring', 'dull', 'weak', 'fragile',
                  'shaky', 'wobbly', 'unstable', 'risky', 'dangerous', 'harmful', 'toxic',
                  'polluted', 'contaminated', 'infected', 'rotten', 'spoiled', 'stale', 'moldy',
                  'dirty', 'dusty', 'messy', 'cluttered', 'chaotic', 'disorganized', 'confused',
                  'lost', 'stuck', 'trapped', 'blocked', 'locked', 'frozen', 'crashed']

# Generate 100 positive reviews
positive_reviews = []
for i in range(100):
    w1 = positive_words[i % len(positive_words)]
    w2 = positive_words[(i + 10) % len(positive_words)]
    templates = [
        f"This product is {w1}",
        f"I {w1} this {w2} product",
        f"Best {w1} ever",
        f"Great {w1} and {w2}",
        f"Highly {w1}"
    ]
    positive_reviews.append(templates[i % 5])

# Generate 100 negative reviews
negative_reviews = []
for i in range(100):
    w1 = negative_words[i % len(negative_words)]
    w2 = negative_words[(i + 10) % len(negative_words)]
    templates = [
        f"This product is {w1}",
        f"I {w1} this {w2} product",
        f"Worst {w1} ever",
        f"Bad {w1} and {w2}",
        f"Very {w1}"
    ]
    negative_reviews.append(templates[i % 5])

# ============================================
# VERIFY & TRAIN
# ============================================

print(f"Positive: {len(positive_reviews)}")
print(f"Negative: {len(negative_reviews)}")

all_reviews = positive_reviews + negative_reviews
all_sentiments = ['positive'] * 100 + ['negative'] * 100

print(f"Total: {len(all_reviews)}")

df = pd.DataFrame({'review': all_reviews, 'sentiment': all_sentiments})

X_train, X_test, y_train, y_test = train_test_split(
    df['review'], df['sentiment'], test_size=0.2, random_state=42, stratify=df['sentiment']
)

model = Pipeline([
    ('tfidf', TfidfVectorizer(lowercase=True, stop_words='english', ngram_range=(1, 2))),
    ('classifier', LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)

train_acc = model.score(X_train, y_train)
test_acc = model.score(X_test, y_test)

print(f"\nTraining Accuracy: {train_acc:.2f}")
print(f"Testing Accuracy: {test_acc:.2f}")

joblib.dump(model, 'sentiment_model.pkl')
print("Model saved!")

# Test
print("\nTesting:")
tests = ['wonderful product', 'terrible waste', 'best thing', 'worst ever']
for text in tests:
    pred = model.predict([text])[0]
    print(f"  '{text}' -> {pred}")

Positive: 100
Negative: 100
Total: 200

Training Accuracy: 1.00
Testing Accuracy: 0.95
Model saved!

Testing:
  'wonderful product' -> positive
  'terrible waste' -> negative
  'best thing' -> positive
  'worst ever' -> negative


In [13]:
from flask import Flask, request, jsonify
import joblib

app = Flask(__name__)
model = joblib.load('sentiment_model.pkl')

@app.route('/')
def home():
    return "Sentiment Analysis API is running!"

@app.route('/predict', methods=['POST'])
def predict():
    data = request.get_json()
    text = data['text']
    sentiment = model.predict([text])[0]
    confidence = model.predict_proba([text])[0].max()

    return jsonify({
        'text': text,
        'sentiment': sentiment,
        'confidence': round(confidence, 2)
    })

print("API built!")

# Test with client
client = app.test_client()

print("\n" + "="*50)
print("TESTING SENTIMENT ANALYSIS API")
print("="*50)

tests = [
    'This product is amazing and I love it',
    'Very bad quality, totally disappointed',
    'Excellent service, highly recommend',
    'Worst purchase ever, complete waste',
    'Okay but not great',
    'Absolutely wonderful experience'
]

for sentence in tests:
    response = client.post('/predict', json={'text': sentence})
    result = response.get_json()
    emoji = '😊' if result['sentiment'] == 'positive' else '😞'
    print(f"\n{emoji} {result['text']}")
    print(f"   Sentiment: {result['sentiment'].upper()}")
    print(f"   Confidence: {result['confidence']*100:.1f}%")

print("\n" + "="*50)
print("ALL TESTS PASSED!")
print("="*50)

API built!

TESTING SENTIMENT ANALYSIS API

😊 This product is amazing and I love it
   Sentiment: POSITIVE
   Confidence: 62.0%

😞 Very bad quality, totally disappointed
   Sentiment: NEGATIVE
   Confidence: 69.0%

😊 Excellent service, highly recommend
   Sentiment: POSITIVE
   Confidence: 70.0%

😞 Worst purchase ever, complete waste
   Sentiment: NEGATIVE
   Confidence: 85.0%

😊 Okay but not great
   Sentiment: POSITIVE
   Confidence: 83.0%

😞 Absolutely wonderful experience
   Sentiment: NEGATIVE
   Confidence: 53.0%

ALL TESTS PASSED!
